# Lecture 20: Introduction to Machine Learning & PyTorch

AY 128; Apr. 2, 2026

# Agenda

- Last CP2 presentations
- What is Machine Learning?
- Supervised Learning: Regression with `sklearn`
- Neural Networks: Concepts & Backpropagation
- Deep Learning Frameworks & PyTorch
- Building an MLP in PyTorch
- Training on Savio (GPU cluster)

# Machine Learning

<img src="https://static1.squarespace.com/static/5150aec6e4b0e340ec52710a/t/51525c33e4b0b3e0d10f77ab/1364352052403/Data_Science_VD.png?format=250w">

http://drewconway.com/zia/2013/3/26/the-data-science-venn-diagram

## What is Machine Learning?

> Field of study that give computers the ability to learn without being explicitly programmed.
> -Arthur Samuel, 1959

<b>Short  Answer</b>:  The offspring of Statistics and Computer Science

<b>Better Answer</b>:  A set of models which aim to learn something about a data set to apply that knowledge to new data

<u>Utility</u>

 - Using labels from training data to classify new objects (e.g., images, digits, webpages)
 - Learning the relationship between explanatory features and response variable to predict for new data (e.g., stock market)
 - Discovering natural clustering structure in data
 - Detecting low-dimensional structure in high-dimensional data
 - Finding outliers in large data sets
 - Game playing/Robotics
 - Chat/coding etc.

 >   *Essentially, all models are wrong, but some are useful.*

 >     -- George Box, Statistician (1919-2013)

## Different Types of Learning

<img src="figs/three.png">

From: [S. Raschka (2015)](https://www.slideshare.net/SebastianRaschka/nextgen-talk-022015/8-Learning_Labeled_data_Direct_feedback)

## Supervised vs. Unsupervised Learning

<img src="figs/learn_types.png">

## Supervised Learning: Regression

Use training set of $(\vec x,y)$ pairs to learn to predict $y$ for new $\vec x$. **Regression** is predicting a *continuous* outcome ($y$) variable from a vector of input features ($\vec x$). That is, we seek to learn:

$f(\vec x) = y$

 In "theory-driven" MCMC modeling, we already think we know from physics what the functional form of $f$ is and what we try to do is figure out the parameters of $f$ that best accommodate the data we have and the beliefs we start with. When we do not know a functional form for $f$ we take more "data driven" approach, such as with Gaussian Processes.

In `sklearn` there are a lot of "data driven" modelling possibilities.

- Linear Regression:  `linear_model.LinearRegression`
- Lasso & Ridge Reg.:  `linear_model.Lasso` / `linear_model.Ridge`
- Gaussian Process Regression: `gaussian_process.GaussianProcess`
- Nearest Neighbor Regression:  `neighbors.KNeighborsRegressor`
- Support Vector Regression:   `svm.SVR`
- Regression Trees:  `tree.DecisionTreeRegressor`

Multioutput regression: $f(\vec x) = \vec y$ and in the case of the last part of Lab 2 we might write this as $f(\vec \theta) = {\rm flux}(\lambda)$ 

### Regression

Let's take a look at the famous California Housing data. We don't have a good physics model for this (of course there are economic theories...). For now we just have data and seek a data-driven model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib import cm
import math

import seaborn as sns
sns.set_context("talk")

from sklearn import datasets
import pandas as pd

%matplotlib inline

In [ ]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

X = housing['data']  # 8 features (e.g. HouseAge,Latitude, AveBedrms, etc.)
Y = housing['target']  # response (median house price in $100,000)

df = pd.DataFrame(X, columns=housing.feature_names)
df["target"]  = housing['target']

# separate out the target into 5 different bins (for viz purposes)
nbins = 5
df["target_binned"] = pd.qcut(df["target"], nbins, labels=False)
df

In [ ]:
print("feature vector shape=", X.shape)
print("output shape=", Y.shape)
print(housing.feature_names)

In [ ]:
f, axs = plt.subplots(1, 3, figsize=(12,6))

for i, ax in enumerate(axs):
    ax.scatter(X[:, i], Y, alpha=0.2, s=2)
    ax.set_xlabel(housing.feature_names[i])
    ax.set_ylabel("Median House Price (in $100,000)")

plt.subplots_adjust(wspace=0.5)

### Basic Model Fitting

We need to create a **training set** and a **testing set**.

In [ ]:
half = math.floor(len(Y)/2)
train_X = X[:half]
train_Y = Y[:half]
test_X = X[half:]
test_Y = Y[half:]

## Linear Regression

The following are a set of methods intended for regression in which the target value is expected to be a linear combination of the input variables. In mathematical notation, if $\hat{y}$ is the predicted value.
$$\hat{y}(w, x) = w_0 + w_1 x_1 + ... + w_p x_p$$
Across the module, we designate the vector $w = (w_1,
..., w_p)$ as `coef_` and $w_0$ as `intercept_`.

http://scikit-learn.org/stable/modules/linear_model.html

In [ ]:
from sklearn import linear_model
from sklearn.metrics import mean_squared_error

clf = linear_model.LinearRegression()

# fit the model
clf.fit(train_X, train_Y)

In [ ]:
# now do the prediction
Y_lr_pred = clf.predict(test_X)

# how well did we do?
mse = mean_squared_error(test_Y, Y_lr_pred)
print(f"MSE = {mse:.4f}")

In [ ]:
f, ax = plt.subplots(figsize=(6, 6))
ax.scatter(test_Y, Y_lr_pred - test_Y, s=2, alpha=0.3)
ax.set_title("Linear Regression Residuals - MSE = %.2f" % mse)
ax.set_xlabel("True Median House Price ($100,000)")
ax.set_ylabel("Residual")
ax.hlines(0, min(test_Y), max(test_Y), color="red")

---
# Neural Networks & Multi-Layer Perceptrons

### What is machine learning with neural networks?

   - A set of methodologies for doing machine learning
   - Collection/composition of simple mathematical functions whose parameterization is *learned* by passing over the data
   - "Deep Learning": modern version of "artificial neural networks"

### Why do people like it?
   - It's "inspired" by how the brain is thought to work, so it *feels* like a natural approach.
   - It works. Amazingly well. In a growing number of use cases.
   - It's composeable, so it's "easy" to understand each piece.
   - Featurizes + learns on "raw" data.
   - Timely: It's tractable with the data/problems we have and the compute power we have access to.
   - New shiny object with codebases getting commoditized (read: easier and easier to use...and free).

### Why do people dislike it?

   - Decades of hype
   - It's considered a black box in a lot of ways
   - It can take a seasoned expert to get it right
   - It's expensive to run/learn a model
   - Not natively adapted to heterogenous data and certain types of learning
   - Not the approach of choice for small/medium data

<img src="http://fastml.com/images/ai/new_navy_device_learns_by_doing.jpg" width="80%">

## The "Neuron" ("Perceptron")
<img src="https://www.evernote.com/l/AUVUbm0I38pMWbUhfC0VUZv7qxxguDOy64QB/image.png">
Source: http://www.wsdm-conference.org/2016/slides/WSDM2016-Jeff-Dean.pdf

<img width="90%" src="https://media.geeksforgeeks.org/wp-content/uploads/20241106171024318092/Artificial-Neural-Networks.webp">

The key is to *learn* the weights $w_i$ given the data. This is done as such:

  1. Initialization:
      - Set the transfer (e.g. sum) & activation (sigmoid) functions you want to use.
      - randomly assign the weights (with some probability distribution)
  2. For each instance $i$, run your input $\vec x_i$ through the network with current weights to get the current output.
  3. Determine $\Delta$ how far off the current output is from the true output/labels.
  4. Update the weights by taking the gradient of the activation at $\vec x_i$ and multiplying by $\Delta$.
  5. Repeat steps 2–5 until you hit a stopping criteria.

This process is an optimization and is called **"Back Propagation"** and, if your activation function is differentiable, it's basically a form of gradient descent and reduces to doing simple linear algebra to find the optimal weights given the data. It was first presented by Rumelhart, Hinton, Williams ([Nature, 1986](http://www.iro.umontreal.ca/~pift6266/A06/refs/backprop_old.pdf))

## Single Layer Network Example

In [ ]:
# input dataset
X_nn = np.array([  [0,0,1],
                   [0,1,1],
                   [1,0,1],
                   [1,1,1] ])

# output label
Y_nn = np.array([[0,0,1,1]]).T
print("X =", X_nn)
print("Y =", Y_nn.T)

In [ ]:
def transfer(wx):
    """
    How to aggregate the weighted inputs.
    Here we'll just do a sum over the weights times x: sum_i w_i x_i
    """
    return np.sum(wx, axis=1)

Different activation functions:

<img src="https://imiloainf.files.wordpress.com/2013/11/activation_funcs1.png" width="50%">

See: https://en.wikipedia.org/wiki/Activation_function

In [ ]:
def activation(twx, func="ReLU", derivative=False):
    """
    Activation: how to treat the sum of the weighted input (twx)
    """
    if func == "ReLU":
        if derivative:
            return np.array([0 if x <= 0 else 1 for x in twx])
        return np.array([max(0, x) for x in twx])

    elif func == "sigmoid":
        if derivative:
            return np.array([x*(1-x) for x in twx])
        return np.array([1/(1+np.exp(-x)) for x in twx])

    elif func == "tanh":
        if derivative:
            np.array([1 - (np.tanh(x))**2 for x in twx])
        return np.array([np.tanh(x) for x in twx])

    else:
        print(f"func {func} not implemented")

In [ ]:
np.random.seed(42)
weights_initial = 2*np.random.random((3,1)) - 1

rms_error = {"tanh": [], "sigmoid": [], "ReLU": []}

for func in ["tanh", "sigmoid", "ReLU"]:

    weights = weights_initial.copy()

    for _ in range(10000):

        # forward propagation
        layer0 = X_nn  # shape = (4,3)
        sum_of_weighted_X = transfer(layer0*weights.T)  # shape = (4,)
        layer1 = activation(sum_of_weighted_X, func=func)  # shape = (4,)

        # how much did we miss?
        layer1_error = Y_nn.T - layer1  # shape = (4,)

        rms_error[func].append(np.sqrt((layer1_error**2).sum()))
        # multiply how much we missed by the
        # slope of the activation at the values in layer1
        layer1_delta = layer1_error * activation(layer1, derivative=True)  # shape = (1,4)
        weights += np.dot(layer1_delta, layer0).T  # shape = (3,1)

In [ ]:
plt.figure(figsize=(6,6))

plt.plot(rms_error["sigmoid"], label="sigmoid")
plt.plot(rms_error["tanh"], label="tanh")
plt.plot(rms_error["ReLU"], label="ReLU")

plt.xscale("symlog")
plt.yscale("log")
plt.ylabel("RMS Error")
plt.xlabel("Generation")
plt.legend()
plt.title("Single Layer NN")

Note we updated the weights at each pass by using all the instances (this is called "batch learning"). There are speed ups (but generally noisier learning) by randomly choosing a subset of the data at each iteration ("stochastic learning").

See [LeCun, Bottou, Orr, & Müller 1998](http://yann.lecun.com/exdb/publis/pdf/lecun-98b.pdf) for more info.

## Multi-Layer Networks

Multilayer networks are not really any different. They have more weights to learn but they may also represent more complex models. Backpropagation optimization still works, this time by using the chain rule. That is, optimization is multi-step but it's local to individual layers (this makes the problem tractable).

<img src="http://scikit-learn.org/stable/_images/multilayerperceptron_network.png" width="50%">

The above network is said to have a hidden layer, which is neither an input nor an output layer.

[Multi-layer neural nets in the Browser](http://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.65948&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false)

# PyTorch

   - syntax closely resembles Python, making it easy for developers familiar with the language to transition to deep learning.
   - Dynamic Computational Graphs:
Unlike some frameworks that require you to define the entire computation graph upfront (static graphs), PyTorch builds the graph on the fly as you execute your code. This makes debugging easier and allows for more flexibility, especially in research settings where rapid prototyping is crucial.
   - PyTorch leverages the power of GPUs to accelerate computations, making it suitable for training large and complex models efficiently.

## PyTorch MLP: California Housing

Let's load up the California housing data again and build a neural network regression model in PyTorch. Note the use of `StandardScaler` to normalize the features — this is important for neural network training.

In [ ]:
from sklearn.preprocessing import StandardScaler

cal_data = datasets.fetch_california_housing()
X = cal_data['data']   # 8 features
Y = cal_data['target'] # response (median house price)

half = math.floor(len(Y)/2)
train_X = X[:half]
train_Y = Y[:half]
test_X = X[half:]
test_Y = Y[half:]

# scale the data; remove mean and scale to unit variance: z = (x - u) / s
# StandardScaler transforms your data such that its distribution will have
# a mean value 0 and standard deviation of 1
scaler = StandardScaler()

# Don't cheat - fit only on training data
scaler.fit(train_X)
train_X = scaler.transform(train_X)

# apply same transformation to test data
test_X = scaler.transform(test_X)

In [ ]:
num_input_features = train_X.shape[1]
print(f'number of input features = {num_input_features}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

print(f"PyTorch version: {torch.__version__}")

Let's make a simple neural network in PyTorch, as an MLP (multi-layer perceptron) with
a few layers.

In [ ]:
class NNClf(nn.Module):
    """
    A simple neural network with three hidden layers and one output layer.
    Uses ReLU activation for the hidden layers.
    """
    def __init__(self, input_features):
        super(NNClf, self).__init__()
        self.layer1 = nn.Linear(input_features, 32)
        self.layer2 = nn.Linear(32, 32)
        self.layer3 = nn.Linear(32, 10)
        self.output_layer = nn.Linear(10, 1)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        x = self.output_layer(x)
        return x

In [ ]:
model = NNClf(num_input_features)
print(model)

In [ ]:
# Convert numpy arrays to PyTorch tensors
train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
train_Y_tensor = torch.tensor(train_Y, dtype=torch.float32).view(-1, 1)  # reshape to a 2D tensor
test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
test_Y_tensor = torch.tensor(test_Y, dtype=torch.float32).view(-1, 1)

print(f"train_X tensor shape: {train_X_tensor.size()}")
print(f"train_Y tensor shape: {train_Y_tensor.size()}")

## Datasets & DataLoaders

"Code for processing data samples can get messy and hard to maintain; we ideally want our dataset code to be decoupled from our model training code for better readability and modularity. PyTorch provides two data primitives: `torch.utils.data.DataLoader` and `torch.utils.data.Dataset` that allow you to use pre-loaded datasets as well as your own data. Dataset stores the samples and their corresponding labels, and DataLoader wraps an iterable around the Dataset to enable easy access to the samples."

https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

In [ ]:
# Create Dataset and DataLoader
train_dataset = TensorDataset(train_X_tensor, train_Y_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
# Define loss and optimizer

# MSE Loss: measures the average squared difference between
# the estimated values and the actual value (good for regression)
criterion = nn.MSELoss()

# Adam optimizer: adaptive learning rate optimization
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training loop
epochs = 50

for epoch in range(epochs):
    model.train()  # Set the model to training mode

    # Iterate over batches of data from the training loader
    for batch_X, batch_Y in train_loader:
        optimizer.zero_grad()    # Clear gradients
        outputs = model(batch_X) # Forward pass
        loss = criterion(outputs, batch_Y)  # Calculate loss
        loss.backward()          # Backward pass (compute gradients)
        optimizer.step()         # Update weights
    print(f"{loss.item():0.3f}", sep=" ", end=" ", flush=True)

In [ ]:
# Evaluate the model
model.eval()  # set ourselves in evaluation mode
with torch.no_grad():
    predictions = model(test_X_tensor).numpy()

# Calculate MSE
mse = np.mean((predictions - test_Y_tensor.numpy()) ** 2)
print("MSE", mse)

In [ ]:
plt.figure(figsize=(10,6))
plt.title("NN Regression Residuals - MSE = %.3f" % mse)
plt.scatter(test_Y_tensor.numpy(), predictions, alpha=0.4, s=3)
plt.xlabel("Test Y")
plt.ylabel("Predicted Y")
plt.plot([0.2, 5], [0.2, 5], c="r")